# Problems & Goal

- Bài toán cốt lõi: Phân loại đơn nhãn (single-label) 6 lớp theo đúng benchmark. Mặc dù thực tế các lớp có sự chồng chéo ngữ nghĩa (ví dụ: ảnh Tết thường bao gồm cảnh tụ họp), ta sẽ xử lý sự mơ hồ này trực tiếp ở khâu huấn luyện và phân tích lỗi thay vì thay đổi định nghĩa bài toán.
- Vai trò của lớp "other": Hoạt động như một phễu lọc (catch-all failure mode) để hứng các mẫu ngoại lai hoặc không rõ ràng, chứ không mang một đặc trưng ngữ nghĩa độc lập.
- Thách thức từ dữ liệu: Kích thước tập mẫu cực nhỏ, mất cân bằng nghiêm trọng, nhãn nhiễu và ranh giới phân loại mờ nhạt (ví dụ: ảnh công viên dễ nhầm thành thiên nhiên).
- Tiêu chí tối ưu: Ưu tiên tính mạnh mẽ (robustness) và độ tin cậy khi đưa vào thực tế. Không chạy đua tối ưu độ chính xác điểm (point accuracy) trên tập dữ liệu nhỏ vốn rất dễ bị overfit.
- Chiến lược thực thi: Tiếp cận theo hướng tối ưu dữ liệu (data-centric) thay vì dùng mạng end-to-end phức tạp. Giải pháp là sử dụng Frozen Embeddings (SigLIP2 đa ngôn ngữ) kết hợp Classification Head có trọng số, đi kèm kỹ thuật hiệu chuẩn (calibration) và lọc nhiễu offline (CLIPCleaner).

# EDA

## Manifest

In [ ]:
from pathlib import Path

from z_photos.conf import Config

cfg = Config(
    data_root=Path("../data"),
    seed=42,
)

## Bronze: FiftyOne Dataset

TODO(Explain why FiftyOne)

In [ ]:
from z_photos.datasets import build_bronze_dataset

bronze_dataset = build_bronze_dataset("z_photos-bronze", bronze_dir=cfg.bronze_dir)

## Inventory & Sanity

1. Class distribution: mất cân bằng giữa train/test?
1. Metadata sanity: kích thước ảnh, file corrupt?
1. Exact duplicates: file trùng lặp chính xác (hash).

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd

# Class distribution per split
train_view = bronze_dataset.match_tags("train")
test_view = bronze_dataset.match_tags("test")

train_counts = Counter(train_view.values("ground_truth.label"))
test_counts = Counter(test_view.values("ground_truth.label"))

all_classes = sorted(set(train_counts) | set(test_counts))
df_dist = pd.DataFrame(
    {
        "class": all_classes,
        "train": [train_counts.get(c, 0) for c in all_classes],
        "test": [test_counts.get(c, 0) for c in all_classes],
    }
).set_index("class")
df_dist["total"] = df_dist["train"] + df_dist["test"]
df_dist["train_pct"] = (df_dist["train"] / df_dist["train"].sum() * 100).round(1)
df_dist["test_pct"] = (df_dist["test"] / df_dist["test"].sum() * 100).round(1)

print("= Class Distribution")
print(f"Train total: {df_dist['train'].sum()}")
print(f"Test total: {df_dist['test'].sum()}")
df_dist.index.name = None
df_dist.style.format({"train_pct": "{:.1f}%", "test_pct": "{:.1f}%"})

In [ ]:
widths = bronze_dataset.values("metadata.width")
heights = bronze_dataset.values("metadata.height")
sizes = bronze_dataset.values("metadata.size_bytes")

missing_meta = sum(1 for w in widths if w is None)
print(f"Samples với metadata thiếu (có thể corrupt): {missing_meta}")

w_arr = np.array([w for w in widths if w], dtype=float)
h_arr = np.array([h for h in heights if h], dtype=float)
s_arr = np.array([s for s in sizes if s], dtype=float)
ar_arr = w_arr / h_arr

df_meta = pd.DataFrame(
    {
        "metric": ["Width (px)", "Height (px)", "Aspect ratio", "File size (KB)"],
        "min": [w_arr.min(), h_arr.min(), ar_arr.min(), s_arr.min() / 1024],
        "max": [w_arr.max(), h_arr.max(), ar_arr.max(), s_arr.max() / 1024],
        "mean": [w_arr.mean(), h_arr.mean(), ar_arr.mean(), s_arr.mean() / 1024],
        "std": [w_arr.std(), h_arr.std(), ar_arr.std(), s_arr.std() / 1024],
    }
).set_index("metric")
df_meta.index.name = None
df_meta.style.format("{:.1f}")

## Exact Duplicates

`fob.compute_exact_duplicates`: so sánh hash file MD5/SHA, phát hiện file trùng chính xác. Không cần model.

In [ ]:
import fiftyone.brain as fob
import pandas as pd

exact_dups = fob.compute_exact_duplicates(bronze_dataset, progress=False)

rows = []
for rep_id, dup_ids in list(exact_dups.items())[:5]:
    rep = bronze_dataset[rep_id]
    for did in dup_ids:
        s = bronze_dataset[did]
        rows.append(
            {
                "Rep Label": rep.ground_truth.label,
                "Rep File": rep.filepath.split("/")[-1],
                "Dup Label": s.ground_truth.label,
                "Dup File": s.filepath.split("/")[-1],
            }
        )

pd.DataFrame(rows)

## SigLIP2: Semantic Index

### Semantic Index

In [ ]:
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from z_photos.fiftyone import register_models

register_models()
CLASSES = sorted(bronze_dataset.distinct("ground_truth.label"))
siglip2_zs = foz.load_zoo_model("z-photos-siglip2-so400m-patch16-384", classes=CLASSES)

unprocessed_view = bronze_dataset.match(~F("siglip2_embeddings").exists())
if len(unprocessed_view) > 0:
    print(f"SigLIP2 embeddings: {len(unprocessed_view)} còn lại")
    unprocessed_view.compute_embeddings(
        siglip2_zs, embeddings_field="siglip2_embeddings"
    )
else:
    print("SigLIP2 embeddings: OK")

In [ ]:
import fiftyone.brain as fob

if "siglip2_viz" in bronze_dataset.list_brain_runs():
    bronze_dataset.delete_brain_run("siglip2_viz")
fob.compute_visualization(
    bronze_dataset, embeddings="siglip2_embeddings", brain_key="siglip2_viz"
)
print("SigLIP2 visualization OK.")

if "siglip2_sim" in bronze_dataset.list_brain_runs():
    bronze_dataset.delete_brain_run("siglip2_sim")
siglip2_sim = fob.compute_similarity(
    bronze_dataset,
    model="z-photos-siglip2-so400m-patch16-384",
    embeddings="siglip2_embeddings",
    brain_key="siglip2_sim",
    backend="qdrant",
)
print(f"SigLIP2 similarity index ok. supports_prompts={siglip2_sim.supports_prompts}")

### Zero-shot Classification

In [ ]:
from collections import Counter

import pandas as pd
from fiftyone import ViewField as F

unprocessed_view = bronze_dataset.match(~F("siglip2_pred").exists())
if len(unprocessed_view) > 0:
    print(f"SigLIP2 zero-shot: {len(unprocessed_view)} còn lại")
    unprocessed_view.apply_model(siglip2_zs, label_field="siglip2_pred")
else:
    print("SigLIP2 zero-shot: OK")

gt_labels = bronze_dataset.values("ground_truth.label")
pred_labels = bronze_dataset.values("siglip2_pred.label")
agree = sum(g == p for g, p in zip(gt_labels, pred_labels, strict=True))
total = len(bronze_dataset)
print(f"\nAgreement: {agree}/{total} = {agree / total * 100:.1f}%")

disagree_by_class = {}
for g, p in zip(gt_labels, pred_labels, strict=False):
    if g != p:
        disagree_by_class.setdefault(g, []).append(p)

rows = []
for gt_cls in sorted(disagree_by_class):
    preds = Counter(disagree_by_class[gt_cls])
    for pred_cls, cnt in preds.most_common():
        rows.append({"ground_truth": gt_cls, "siglip2_pred": pred_cls, "count": cnt})

df_disagree = pd.DataFrame(rows)
df_disagree.set_index(["ground_truth", "siglip2_pred"])

   0% ||----------------|   1/321 [51.4s elapsed, 4.6h remaining, 0.0 samples/s] 

/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/fiftyone/utils/torch.py:1356: RuntimeWarning: invalid value encountered in divide
  odds /= np.sum(odds, axis=1, keepdims=True)


   1% |-----------------|   3/321 [51.8s elapsed, 1.5h remaining, 0.1 samples/s] 

/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/fiftyone/utils/torch.py:1355: RuntimeWarning: overflow encountered in exp
  odds = np.exp(logits)


 100% |█████████████████| 321/321 [1.8m elapsed, 0s remaining, 5.9 samples/s]      

SigLIP2 zero-shot vs ground_truth agreement: 39/321 = 12.1%


count
ground_truth siglip2_pred         
baby_playing lunar_new_year     48
gathering    lunar_new_year     29
nature       lunar_new_year     74
other        lunar_new_year     80
             baby_playing        2
trekking     lunar_new_year     49

## C-RADIOv4: Visual QA Index

C-RADIOv4 là **multi-teacher distilled model** (SigLIP2 + DINOv3 + SAM3) → embedding rất mạnh về visual detail.

Dùng cho:
- `compute_near_duplicates` — ảnh **rất giống nhau về thị giác** (crop, resize, re-composition)
- `compute_leaky_splits` — phát hiện data leakage train → test
- `compute_uniqueness` — phân tích cụm thị giác

In [ ]:
import fiftyone.zoo as foz
from fiftyone import ViewField as F

foz.register_zoo_model_source("https://github.com/harpreetsahota204/CRADIOv4")
radio_model = foz.load_zoo_model("nv_labs/c-radio_v4-so400m")

unprocessed_view = bronze_dataset.match(~F("radio_embeddings").exists())
if len(unprocessed_view) > 0:
    print(f"RADIO embeddings: {len(unprocessed_view)} còn lại")
    unprocessed_view.compute_embeddings(
        radio_model, embeddings_field="radio_embeddings", num_workers=0
    )
else:
    print("RADIO embeddings: OK")

 100% |█████████████████| 321/321 [4.1m elapsed, 0s remaining, 1.3 samples/s]      


### Near Duplicates

In [ ]:
# Build similarity index từ RADIO embeddings (dùng để near-dups + leaky splits)
radio_sim = fob.compute_similarity(
    bronze_dataset,
    embeddings="radio_embeddings",
    brain_key="radio_sim",
    backend="qdrant",
)

# Near duplicates: ảnh rất giống nhau về mặt thị giác
near_dup_index = fob.compute_near_duplicates(
    bronze_dataset,
    similarity_index=radio_sim,
    threshold=0.1,  # cosine distance < 0.1 → very similar visually
)

dup_ids = near_dup_index.duplicate_ids
print(f"Near duplicate samples (C-RADIOv4, threshold=0.1): {len(dup_ids)}")
rate = len(dup_ids) / len(bronze_dataset) * 100
print(f"Near-dup rate: {len(dup_ids)}/{len(bronze_dataset)} = {rate:.1f}%")

if dup_ids:
    dup_samples = bronze_dataset.select(dup_ids)
    dup_label_counts = Counter(dup_samples.values("ground_truth.label"))
    dup_tag_counts = Counter([t for ts in dup_samples.values("tags") for t in ts])
    df_nd = pd.DataFrame(
        {
            "class": list(dup_label_counts.keys()),
            "near_dup_count": list(dup_label_counts.values()),
        }
    ).set_index("class")
    display(df_nd.style.format("{:d}"))

# Future Works